# P40 — Dropout: una forma simple de evitar el sobreajuste en redes neuronales

## 1. Título y paper

**Paper:** *Dropout: A Simple Way to Prevent Neural Networks from Overfitting*  
**Autoría:** Nitish Srivastava, Geoffrey Hinton, Alex Krizhevsky, Ilya Sutskever, Ruslan Salakhutdinov  
**Año y venue:** 2014 · JMLR 15(56):1929–1958  
**Nivel:** L2 · **Motor:** `dropout`  
**Ficha completa:** [`P40_dropout`](../../papers/foundational/P40_dropout/README.md)

**Hito:** Apagar unidades al azar durante el entrenamiento equivale a entrenar un ensamblado exponencial de subredes que comparten pesos.

- [JMLR 15(56)](https://jmlr.org/papers/v15/srivastava14a.html)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Las redes grandes memorizaban el conjunto de entrenamiento, y las unidades desarrollaban co-adaptaciones frágiles: una función solo servía si su 'socia' estaba presente.
2. Ejecutar una implementación mínima de la propuesta: En cada paso, poner a cero cada unidad con probabilidad p. Ninguna función puede depender de una unidad concreta, así que la red aprende representaciones redundantes.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P02
- P04


## 4. Intuición

Si en un equipo cada tarea depende de dos personas concretas, el día que una falte no se hace nada. Si todos pueden cubrir varias tareas, el equipo aguanta. Dropout obliga a lo segundo haciendo faltar a gente al azar cada día.


## 5. Concepto mínimo

```text
Entrenamiento:  h̃ = h ⊙ m,   m ~ Bernoulli(1−p)
Inferencia   :  se usan todas, escaladas por (1−p)

Con n unidades hay 2ⁿ subredes posibles, todas compartiendo pesos.
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('dropout', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Con qué probabilidad están activas a la vez dos unidades concretas, si p=0,5?
2. ¿Y al menos una de tres?
3. ¿Qué tipo de representación premia eso?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('dropout', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('dropout', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

Una función que depende de dos unidades concretas solo está disponible una cuarta parte de las veces; una repartida entre tres, casi siempre. Dropout no «añade ruido»: **cambia qué representaciones son rentables**.


## 10. Comentario pedagógico

Dropout dominó la regularización durante años y hoy se usa mucho menos en visión y en Transformers grandes, donde otras técnicas y la escala de datos cumplen ese papel. Es un buen recordatorio de que las recetas caducan.


## 11. Error o anti-patrón deliberado

Anti-patrón: dejar dropout activo en inferencia.


In [ ]:
print('Con dropout activo en inferencia, la misma entrada da salidas distintas.')
print('Y la magnitud de las activaciones es (1-p) veces la esperada.')
print('Es un fallo clasico: el modelo "funciona peor en produccion" sin causa aparente.')

## 12. Corrección

La corrección es la escala, y el motivo es que la esperanza cuadre:


In [ ]:
p = 0.5
print('en entrenamiento: se apaga la fraccion p, la suma esperada baja a (1-p) del total')
print(f'en inferencia   : o se multiplica por (1-p)={1-p}, o se divide en entrenamiento')
print('las dos convenciones existen; usar las dos a la vez rompe el modelo')

## 13. Desafío guiado

Calcula la probabilidad de que una función que depende de 4 unidades concretas esté disponible.


In [ ]:
r = run_paper_lab('dropout', seed=3)['result']
show(r)

## 14. Desafío autónomo

Entrena una red pequeña con y sin dropout sobre un conjunto con pocos datos. Compara la brecha entre error de entrenamiento y de validación, no solo el error final.


## 15. Evidencia de aprendizaje

Guarda las probabilidades de disponibilidad, el conteo de subredes y tu explicación del escalado en inferencia.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P40_dropout/README.md) · evaluación formal: [`assessments/papers/P40_dropout.md`](../../assessments/papers/P40_dropout.md)


## 16. Cierre

La red ya no memoriza. Pero sigue siendo difícil de optimizar si cada dirección tiene una curvatura distinta.


## 17. Conexión con el siguiente hito

- P43
- regularización moderna

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
